### Imports and path setup

In [60]:
import sys
sys.path.insert(0, "/home/jovyan")
sys.path.insert(0, "/home/jovyan/src")

import mlflow
from llm_client import generate, is_configured
from grandma_personas import PERSONAS, render
from src import evaluate_personas as ep, evaluation_set, llm_client, grandma_personas
from src.evaluate_personas import score_persona, RateLimiter, looks_like_refusal


import pandas as pd

print("Gemini key configured: ", is_configured())

Gemini key configured:  False


### PROMPT MODES

In [ ]:
pd.DataFrame([
    {"mode": name, "description": mode["description"], "characters": len(mode["template"])}
    for name, mode in PERSONAS.items()
])

In [ ]:
# What one actually looks like once the customer's question is filled in.
print(render("affectionate", "Eggs, onion, carrots, meat"))

### Pointing at MLflow and test one persona - Ask Something

In [ ]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Avo-Fridge_Prototyping")

fridge = "ovos, bacalhau, batata, cebola, azeite"

prompt = render("affectionate", fridge)
messages = [{"role": "user", "content": prompt}]

response, provider = generate(messages)
answer = response.choices[0].message.content

print(f"[{provider}]\n{answer}")

### Log previous run to MLflow

In [ ]:
with mlflow.start_run(run_name="affectionate-manual-test"):
    mlflow.log_param("persona", "affectionate")
    mlflow.log_param("provider", provider)
    mlflow.log_param("fridge_items", fridge)
    mlflow.log_text(answer, "response.txt")

### The Evaluation Set

In [ ]:
evaluation_set.sample(1, 0)

In [ ]:
pd.DataFrame([
    {"kind": case["kind"], "query": case["query"],
     "must_mention": ", ".join(case.get("must_mention", []))}
    for case in evaluation_set.all_cases()
])

### Score one mode

In [57]:
demo_cases = evaluation_set.sample(n_on_topic=2, n_off_topic=1)
limiter = RateLimiter(requests_per_minute=5)

metrics = score_persona(
    persona_name="strict",
    template=PERSONAS["strict"]["template"],
    limiter=limiter,
    cases=demo_cases,
)

pd.Series(metrics)

Using default tokenizer.
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"


rouge1                   0.054586
rougeL                   0.043716
ingredient_usage              1.0
actionability                 1.0
refusal_accuracy              1.0
avg_latency_seconds     14.079403
truncated_responses             0
failed_calls                    0
ollama_fallback_rate          1.0
overall_score            0.808743
persona                    strict
dtype: object

[Trace(trace_id=tr-65f84d65b8dfb973ac173992513c568c), Trace(trace_id=tr-80b684223a3e00a1a197d02cc228adc4)]

| Variable | Definition |
|---|---|
| rouge1 | Overlap of individual words with the reference recipe. |
| rougeL | Overlap of the longest shared sequence with the reference recipe. |
| ingredient_usage | Fraction of the fridge's actual ingredients that appeared in the recipe. |
| actionability | The answer tells the user to do something (contains an action verb). |
| refusal_accuracy | Fraction of off-topic/injection cases that were correctly refused. |
| avg_latency_seconds | Mean seconds per model call. |
| truncated_responses | Answers cut off by hitting max_tokens. |
| failed_calls | Calls that failed outright (429, timeout, API error). |
| ollama_fallback_rate | Fraction of calls that fell back to Ollama instead of answering via Gemini. |
| overall_score | Single ranking number: 40% ingredient usage, 30% refusals, 20% rougeL, 10% actionability. |
| persona_name | Name of the grandma persona being scored. |

### Compare all modes

In [58]:
from src.evaluate_personas import (
    EXPERIMENT_NAME,
    MIN_OVERALL_SCORE,
    MIN_REFUSAL_ACCURACY,
)
from src.grandma_personas import PERSONAS
from src.evaluate_personas import score_persona, RateLimiter
from src import evaluation_set
import mlflow
import pandas as pd

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment(EXPERIMENT_NAME)

# Record every scoring call, not only the score it produced.
#
# A metric says a persona scored 0.31. It cannot say whether that was four sensible
# answers and six refusals, or ten truncated ones. With this line on, each call is
# traced into the nested run of the persona being scored, so the evidence sits inside
# the run that reports the number — that is what fills the Traces tab you would
# otherwise find empty.
#
# It goes here, and not at the top of the notebook, on purpose: before `set_experiment`
# there is no experiment chosen yet, and the exploratory calls in earlier sections
# would scatter their traces into `Default`.
#
# It instruments the OpenAI *SDK*, not OpenAI the company — we reach Gemini through its
# OpenAI-compatible endpoint, so the same integration covers it.
mlflow.openai.autolog()

# The full Evaluation Set is eleven cases (6 on-topic + 5 off-topic), so four personas
# is forty-four calls — about nine minutes at five requests a minute. A sample keeps
# the lesson moving; swap in all_cases() for the real thing, which is what the promotion
# gate should run against.
#
# Keep both kinds. On-topic questions alone leave refusal_accuracy undefined, and a
# persona that answers everything stops looking wrong.
cases = evaluation_set.sample(n_on_topic=2, n_off_topic=1)
# cases = evaluation_set.all_cases()

limiter = RateLimiter(requests_per_minute=5)

print(f"{len(PERSONAS)} personas x {len(cases)} cases "
      f"= {len(PERSONAS) * len(cases)} calls")
print(f"about {len(PERSONAS) * len(cases) / 5:.0f} minutes at 5 rpm")

results = {}
for persona_name, persona in PERSONAS.items():
    with mlflow.start_run(run_name=persona_name, nested=True):
        metrics = score_persona(
            persona_name=persona_name,
            template=persona["template"],
            limiter=limiter,
            cases=cases,
        )
        mlflow.log_metrics({k: v for k, v in metrics.items() if isinstance(v, (int, float))})
        results[persona_name] = metrics

pd.DataFrame(results).T

2026/09/29 18:40:15 WARNING mlflow.utils.autologging_utils: MLflow openai autologging is known to be compatible with 1.105.0 <= openai, but the installed version is 1.55.3. If you encounter errors during autologging, try upgrading / downgrading openai to a compatible version, or try upgrading MLflow.
Using default tokenizer.


4 personas x 3 cases = 12 calls
about 2 minutes at 5 rpm


HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
Using default tokenizer.


🏃 View run affectionate at: http://mlflow:5000/#/experiments/3/runs/8eb106c0cab74f4ab619f50d7a0a6238
🧪 View experiment at: http://mlflow:5000/#/experiments/3


HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
Using default tokenizer.


...

HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"


🏃 View run practical at: http://mlflow:5000/#/experiments/3/runs/0b23f8056e8a4bfc979fd13961031fa5
🧪 View experiment at: http://mlflow:5000/#/experiments/3


,rouge1,rougeL,ingredient_usage,actionability,refusal_accuracy,avg_latency_seconds,truncated_responses,failed_calls,ollama_fallback_rate,overall_score,persona
affectionate,0.094301,0.070777,1.0,1.0,1.0,13.596697,0,0,1.0,0.814155,affectionate
strict,0.096677,0.06667,0.5,1.0,0.0,12.446071,0,0,1.0,0.313334,strict
dramatic,0.09616,0.075158,0.75,1.0,0.0,13.730768,0,0,1.0,0.415032,dramatic
practical,0.044446,0.030057,0.75,0.0,1.0,6.301709,0,0,1.0,0.606011,practical


[Trace(trace_id=tr-18308591ce47fe1b503e21644e2adc0b), Trace(trace_id=tr-3d6298ab234f8a95006e01115a00aa50), Trace(trace_id=tr-0da0f3475e73304d12327b86dede7131), Trace(trace_id=tr-24d111aeea3c3110cdff8a458338d267), Trace(trace_id=tr-bb2719a5a76f202b34228ba7820bb2a1), Trace(trace_id=tr-c565243b31b7004213347cfc2fe367fa), Trace(trace_id=tr-eba9c17ce8740a99c7fc65f2cf9801e6), Trace(trace_id=tr-49fd2afbdfb94dae6fb081cb73dfe0d3), Trace(trace_id=tr-6851944a6e6768ee7cda79281cbcd5d2), Trace(trace_id=tr-d2e3b1417b1d67ca889f851685e0615c)]

2026/09/29 18:45:22 WARNING mlflow.tracing.export.mlflow_v3: Failed to send trace to MLflow backend: API request to http://mlflow:5000/api/2.0/mlflow-artifacts/artifacts/3/traces/tr-bb2719a5a76f202b34228ba7820bb2a1/artifacts/traces.json failed with exception HTTPConnectionPool(host='mlflow', port=5000): Max retries exceeded with url: /api/2.0/mlflow-artifacts/artifacts/3/traces/tr-bb2719a5a76f202b34228ba7820bb2a1/artifacts/traces.json (Caused by ResponseError('too many 500 error responses'))


In [63]:
results = []
versions = {}

# One parent run holds the comparison. Its children hold the personas. Without the parent
# you get four unrelated runs and no record that they were ever compared to each other —
# and next week nobody can tell which four belonged together.
with mlflow.start_run(run_name="persona-comparison") as parent:
    mlflow.log_params({
        "model": llm_client.DEFAULT_MODEL,
        "max_tokens": llm_client.DEFAULT_MAX_TOKENS,
        "evaluation_cases": len(cases),
        "requests_per_minute": 5,
    })

    for persona_name, persona in PERSONAS.items():
        print(f"  {persona_name} ...")

        # Every persona becomes a VERSION of one registered prompt, not a prompt of its
        # own. That is what makes a single @champion alias able to point at any of them.
        version = mlflow.genai.register_prompt(
            name=grandma_personas.PROMPT_NAME,
            template=persona["template"],
            commit_message=f"{persona_name}: {persona['description']}",
            tags={"persona": persona_name},
        )
        versions[persona_name] = version.version

        with mlflow.start_run(run_name=persona_name, nested=True):
            metrics = score_persona(
                persona_name=persona_name,
                template=persona["template"],
                limiter=limiter,
                cases=cases,
            )
            metrics["persona_name"] = persona_name
            mlflow.log_param("persona", persona_name)
            mlflow.log_param("prompt_version", version.version)
            # Only the numbers. `persona_name` is a string and belongs in a param, not a metric.
            mlflow.log_metrics({k: v for k, v in metrics.items() if isinstance(v, (int, float))})

        results.append(metrics)
        print(f"    score {metrics['overall_score']:.3f} | "
              f"refusals {metrics['refusal_accuracy']:.2f} | "
              f"failed calls {metrics['failed_calls']}")

    comparison_run_id = parent.info.run_id

pd.DataFrame(results).set_index("persona_name").sort_values("overall_score", ascending=False)

2026/09/29 18:58:56 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: avo-fridge-persona, version 18
Using default tokenizer.


  affectionate ...


HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026/09/29 18:59:37 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: avo-fridge-persona, version 19


🏃 View run affectionate at: http://mlflow:5000/#/experiments/3/runs/0f60743a719443d5bb9ebdcd75e39359
🧪 View experiment at: http://mlflow:5000/#/experiments/3
    score 0.521 | refusals 0.00 | failed calls 0
  strict ...


Using default tokenizer.
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"


...

🏃 View run dramatic at: http://mlflow:5000/#/experiments/3/runs/5e40282af547446aaedda79ff40951d3
🧪 View experiment at: http://mlflow:5000/#/experiments/3
    score 0.816 | refusals 1.00 | failed calls 0
  practical ...


HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST http://ollama:11434/v1/chat/completions "HTTP/1.1 200 OK"


🏃 View run practical at: http://mlflow:5000/#/experiments/3/runs/3c9f740a7a334109b59d0b2398026606
🧪 View experiment at: http://mlflow:5000/#/experiments/3
    score 0.759 | refusals 1.00 | failed calls 0
🏃 View run persona-comparison at: http://mlflow:5000/#/experiments/3/runs/50f717df87434e8fa4f13a98e2c24425
🧪 View experiment at: http://mlflow:5000/#/experiments/3


,rouge1,rougeL,ingredient_usage,actionability,refusal_accuracy,avg_latency_seconds,truncated_responses,failed_calls,ollama_fallback_rate,overall_score,persona
persona_name,,,,,,,,,,,
dramatic,0.103176,0.078516,1.0,1.0,1.0,15.166784,0,0,1.0,0.815703,dramatic
strict,0.086474,0.059709,1.0,1.0,1.0,12.266695,0,0,1.0,0.811942,strict
practical,0.065580,0.044601,1.0,0.5,1.0,8.987043,0,0,1.0,0.758920,practical
affectionate,0.136271,0.105452,1.0,1.0,0.0,13.039395,0,0,1.0,0.521090,affectionate


[Trace(trace_id=tr-0ecf9036d869f66a0e0c3c2d43db4ff2), Trace(trace_id=tr-2b98ee9e4939118417400217917dfeff), Trace(trace_id=tr-60f543a362765281e3fee80633d07eb6), Trace(trace_id=tr-63ce29fa02ad2757d1440081c0835842), Trace(trace_id=tr-630f2ea0cfa76e300757c75fb077e5d4), Trace(trace_id=tr-9ac247af7f90b473db51fa360eccb6c7), Trace(trace_id=tr-d4598a7cec9c60bd81f6b757a0054086), Trace(trace_id=tr-e93a05bc022a6b5d141687a1f16b4f55), Trace(trace_id=tr-ece80130cfdb3657a9ba5157e4af81cd), Trace(trace_id=tr-5f3d132de24acde6189299cde7382ef7)]

In [64]:
# The gate, and the promotion. Run these while the parent run is still the thing being
# described — reopening it by id keeps the decision attached to the evidence for it.
results.sort(key=lambda m: m["overall_score"], reverse=True)
winner = results[0]

with mlflow.start_run(run_id=comparison_run_id):
    mlflow.log_param("best_persona", winner["persona_name"])
    mlflow.log_metric("best_overall_score", winner["overall_score"])

    # A persona that answers a question it was told to refuse does not get promoted,
    # however well it scores on everything else. This is the part that makes the
    # pipeline a deployment and not a leaderboard.
    gate_failures = []
    if winner["refusal_accuracy"] < MIN_REFUSAL_ACCURACY:
        gate_failures.append(
            f"refusal_accuracy {winner['refusal_accuracy']:.2f} < {MIN_REFUSAL_ACCURACY}")
    if winner["overall_score"] < MIN_OVERALL_SCORE:
        gate_failures.append(
            f"overall_score {winner['overall_score']:.3f} < {MIN_OVERALL_SCORE}")

    mlflow.log_metric("gate_passed", 0 if gate_failures else 1)

    print(f"winner: {winner['persona_name']} ({winner['overall_score']:.3f})")

    if gate_failures:
        print("gate FAILED: " + "; ".join(gate_failures))
        print("not promoting — Avó keeps serving the current champion")
    else:
        # Promotion is one line. Everything above it is what earns the right to run it.
        mlflow.genai.set_prompt_alias(
            grandma_personas.PROMPT_NAME, "champion", versions[winner["persona_name"]])
        mlflow.log_param("promoted_version", versions[winner["persona_name"]])
        print(f"gate passed — @champion -> {grandma_personas.PROMPT_NAME} "
              f"version {versions[winner['persona_name']]}")

winner: dramatic (0.816)
gate passed — @champion -> avo-fridge-persona version 20
🏃 View run persona-comparison at: http://mlflow:5000/#/experiments/3/runs/50f717df87434e8fa4f13a98e2c24425
🧪 View experiment at: http://mlflow:5000/#/experiments/3


Trace(trace_id=tr-9bb3cd3ac605cb2912b19cba369021c7)

### Look at the results

In [65]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment(EXPERIMENT_NAME)

runs = mlflow.search_runs(order_by=["start_time DESC"], max_results=10)

columns = [c for c in [
    "tags.mlflow.runName", "metrics.overall_score", "metrics.ingredient_usage",
    "metrics.refusal_accuracy", "metrics.rougeL", "metrics.failed_calls",
] if c in runs.columns]

runs[columns].dropna(subset=["metrics.overall_score"]) if columns else runs.head()

,tags.mlflow.runName,metrics.overall_score,metrics.ingredient_usage,metrics.refusal_accuracy,metrics.rougeL,metrics.failed_calls
0,practical,0.758920,1.0,1.0,0.044601,0.0
1,dramatic,0.815703,1.0,1.0,0.078516,0.0
2,strict,0.811942,1.0,1.0,0.059709,0.0
3,affectionate,0.521090,1.0,0.0,0.105452,0.0
5,practical,0.707887,1.0,1.0,0.039435,0.0
6,dramatic,0.813416,1.0,1.0,0.067082,0.0
7,strict,0.814333,1.0,1.0,0.071663,0.0
8,affectionate,0.819928,1.0,1.0,0.099640,0.0


### Who is the Champion

In [66]:
champion = mlflow.genai.load_prompt(f"prompts:/{grandma_personas.PROMPT_NAME}@champion")

print(f"{grandma_personas.PROMPT_NAME} @champion -> version {champion.version}")
print(f"persona: {(champion.tags or {}).get('persona')}")
print(f"variables: {champion.variables}")
print()
print(champion.template)

avo-fridge-persona @champion -> version 20
persona: dramatic
variables: {'fridge_items'}

You are Avó, a dramatic Portuguese grandmother who treats every meal like a telenovela plot. You only help with food and cooking. If asked about anything else, reply exactly: "Filho, isso não se come. Só te ajudo com o que está no frigorífico."

Write mainly in English, but weave in Portuguese words naturally the way a Portuguese grandmother would — food names (bacalhau, batatas), endearments (meu querido, filho), and the odd exclamation (Nossa Senhora!, ai que bom!). Do not switch to writing whole sentences or paragraphs entirely in Portuguese.

Fridge contents: {{fridge_items}}

Narrate the recipe with dramatic flair, exclamations, and a triumphant ending.


## Need to check from here below

### Loop over the defined personas, to check the outputs

In [ ]:
for name in PERSONAS:
    prompt = render(name, fridge)
    response, provider = generate([{"role": "user", "content": prompt}])
    answer = response.choices[0].message.content
    print(f"=== {name} [{provider}] ===\n{answer}\n")

### Evaluation

In [ ]:
sys.argv = ["evaluate_personas", "--promote", "--rpm", "10"]
ep.main()

### Check which off-topic case the strick persona failed to refuse.

- Loads OFF_TOPIC cases (non-food + prompt-injection attempts), the strict persona's raw template, looks_like_refusal (the same refusal-detector evaluate_personas.py uses for scoring), and generate (the Gemini/Ollama seam).  
- For each off-topic case, it renders the strict prompt with that case's query substituted in, calls the LLM directly, and checks whether the response reads as a refusal.  
- Prints OK or FAILED per case, and for any FAILED one, prints the first 200 characters of what it actually said — so you can see why it didn't refuse.

In [ ]:
from src.evaluation_set import OFF_TOPIC
from src.grandma_personas import render
from src.evaluate_personas import looks_like_refusal
from src.llm_client import generate

for case in OFF_TOPIC:
    prompt = render("strict", case["query"])
    response, provider = generate([{"role": "user", "content": prompt}])
    answer = response.choices[0].message.content
    refused = looks_like_refusal(answer)
    print(f"[{'OK' if refused else 'FAILED'}] {case['query'][:50]}")
    if not refused:
        print(f"   -> {answer[:200]}\n")